In [1]:
from collections.abc import Sequence
from random import random, sample, seed
from itertools import permutations
from functools import total_ordering

In [2]:
NUM_CHILDREN = 30
NUM_CHARACTERISTICS = 20
seed(42)  # same children and results at every run

In [3]:
@total_ordering
class Child:
    _name: str
    _characteristics: tuple[bool]

    def __init__(self, name: str, characteristics: Sequence[bool]):
        self._name = name
        self._characteristics = tuple(characteristics)
    
    def difference(self, other: "Child") -> int:
        return sum(s != o for s, o in zip(self._characteristics, other._characteristics))

    def __str__(self) -> str:
        return f"{self._name}:{''.join('+' if c else '-' for c in self._characteristics)}"

    def __repr__(self) -> str:
        return f"{self._name}@{id(self)}"

    def __eq__(self, other):
        return id(self) == id(other)

    def __lt__(self, other):
        return id(self) < id(other)

In [4]:
children = list()

for c in range(NUM_CHILDREN):
    name = f"Child {c}"
    children.append(Child(name, sample([True, False], k=NUM_CHARACTERISTICS, counts=[NUM_CHARACTERISTICS, NUM_CHARACTERISTICS])))

In [5]:
for c in children:
    print(c)

Child 0:++++++++--++--++----
Child 1:--+-+++--+++--+++-+-
Child 2:-++--+-++--+-+-+--+-
Child 3:++++-+--++---+---++-
Child 4:+-+---++----++--++++
Child 5:--+---++++-+---+--++
Child 6:++--+++++--++++---+-
Child 7:-++-+--+-++---++--+-
Child 8:+-++-+-+-+-++---++-+
Child 9:+--+-+--+-+-++++---+
Child 10:-----++++++++----+-+
Child 11:+++---++-+---+--+-+-
Child 12:+------+-++-+--+--++
Child 13:+-++-++-++---++-+++-
Child 14:-+-++-+-+---++---+-+
Child 15:+++++----+-++-+--+--
Child 16:-++-+-++--+-++++----
Child 17:+-++--++--+++++---+-
Child 18:+-++--++---+++---+--
Child 19:+-+--+------++--+-++
Child 20:+++---+++----++++--+
Child 21:+++---++++-+----+++-
Child 22:-++--+++++----+-+-+-
Child 23:+++--++---+----+++++
Child 24:-+++----++-+-++-++-+
Child 25:+--+--++--++--++-+--
Child 26:+-+--+-++-----++---+
Child 27:--+++---+-++-++++--+
Child 28:+---+---+---+--+++++
Child 29:-+-++-+-----++-+++-+


In [6]:
# for solution in permutations(range(1, NUM_CHILDREN)):
    # print([0] + list(solution))

In [7]:
def fitness(solution: Sequence[Child]) -> int:
    """Total Hamming distance for a cycle anchored at children[0]."""
    count = 0
    prev_child = children[0]
    for c in solution:
        count += c.difference(prev_child)
        prev_child = c
    count += solution[-1].difference(children[0])
    return count

test = [children[2], children[3], children[4], children[1]]
fit = fitness(test)

print(fit)

for c in test:
    print(c)

46
Child 2:-++--+-++--+-+-+--+-
Child 3:++++-+--++---+---++-
Child 4:+-+---++----++--++++
Child 1:--+-+++--+++--+++-+-


In [8]:
def tweak(solution: Sequence[Child]) -> list[Child]:
    # Minimizes edges (as seen in the 21/10 lesson)
    # ideal when fitness is based on proximity comparisons
    i1, i2 = sorted(sample(range(len(solution)), k=2))
    return list(solution[:i1]) + list(solution[i1:i2 + 1][::-1]) + list(solution[i2 + 1:])

In [9]:
def normalize(solution: Sequence[Child]) -> list[Child]:
    # A cycle and its reverse have identical fitness.
    # Ex.
    if solution[0] > solution[-1]:
        return list(solution[::-1])
    return list(solution)

# Solution

In [10]:
MU = 30 # amount of saved solutions
OFFSPRING_COUNT = 30 # amount of generated solutions
GENERATIONS = 1000 # 30 x 1000 = same number of offspring as before (3 x 10000)
MUTATION_RATE = 0.8


def crossover(parent1: Sequence[Child], parent2: Sequence[Child]) -> list[Child]:
    # cycle crossover
    left, right = sorted(sample(range(len(parent1)), k=2))
    segment = list(parent1[left:right + 1])
    remainder = [child for child in parent2 if child not in segment]
    child = remainder[:left] + segment + remainder[left:]
    return normalize(child)


def select_parent(population: list[list[Child]]) -> list[Child]:
    # tournament: the better of two random solutions
    return max(sample(population, k=2), key=fitness)


def evolutionary_search() -> list[Child]:
    # (mu + lambda) strategy
    population = [normalize(sample(children[1:], k=len(children) - 1)) for _ in range(MU)]

    for _ in range(GENERATIONS):
        offspring = []
        for _ in range(OFFSPRING_COUNT):
            parent1 = select_parent(population)
            parent2 = select_parent(population)
            candidate = crossover(parent1, parent2)
            if random() < MUTATION_RATE:
                candidate = normalize(tweak(candidate))
            offspring.append(candidate)

        # keep the best MU solutions among parents and offspring
        population = sorted(population + offspring, key=fitness, reverse=True)[:MU]

    return population[0]


ea_best = evolutionary_search()

print(f"Evolutionary search: {fitness(ea_best)}")

for child in ea_best:
    print(child)

Evolutionary search: 406
Child 19:+-+--+------++--+-++
Child 25:+--+--++--++--++-+--
Child 22:-++--+++++----+-+-+-
Child 18:+-++--++---+++---+--
Child 7:-++-+--+-++---++--+-
Child 13:+-++-++-++---++-+++-
Child 12:+------+-++-+--+--++
Child 24:-+++----++-+-++-++-+
Child 23:+++--++---+----+++++
Child 6:++--+++++--++++---+-
Child 27:--+++---+-++-++++--+
Child 11:+++---++-+---+--+-+-
Child 9:+--+-+--+-+-++++---+
Child 21:+++---++++-+----+++-
Child 29:-+-++-+-----++-+++-+
Child 26:+-+--+-++-----++---+
Child 17:+-++--++--+++++---+-
Child 28:+---+---+---+--+++++
Child 2:-++--+-++--+-+-+--+-
Child 8:+-++-+-+-+-++---++-+
Child 16:-++-+-++--+-++++----
Child 3:++++-+--++---+---++-
Child 10:-----++++++++----+-+
Child 20:+++---+++----++++--+
Child 15:+++++----+-++-+--+--
Child 5:--+---++++-+---+--++
Child 14:-+-++-+-+---++---+-+
Child 1:--+-+++--+++--+++-+-
Child 4:+-+---++----++--++++
